In [2]:
import pandas as pd

# Load datasets
yield_df = pd.read_csv("crop_yield.csv")
weather_df = pd.read_csv("weather.csv")
soil_df = pd.read_csv("soil.csv")

# -----------------------------
# STEP 1: CLEAN COLUMN NAMES
# -----------------------------
yield_df.columns = yield_df.columns.str.lower().str.strip()
weather_df.columns = weather_df.columns.str.lower().str.strip()
soil_df.columns = soil_df.columns.str.lower().str.strip()

# -----------------------------
# STEP 2: RENAME FOR CONSISTENCY
# (Adjust based on your dataset)
# -----------------------------
yield_df = yield_df.rename(columns={
    'state': 'location',
    'year': 'year',
    'yield': 'crop_yield'
})

weather_df = weather_df.rename(columns={
    'state': 'location',
    'year': 'year',
    'rainfall': 'rainfall_mm',
    'temperature': 'temp_c'
})

soil_df = soil_df.rename(columns={
    'state': 'location',
    'ph': 'soil_ph'
})

# -----------------------------
# STEP 3: MERGE YIELD + WEATHER
# -----------------------------
merged_df = pd.merge(
    yield_df,
    weather_df,
    on=['location', 'year'],
    how='inner'   # use 'outer' if you want all data
)

# -----------------------------
# STEP 4: MERGE WITH SOIL DATA
# -----------------------------
final_df = pd.merge(
    merged_df,
    soil_df,
    on='location',   # soil may not have year
    how='left'
)

# -----------------------------
# STEP 5: HANDLE MISSING VALUES
# -----------------------------
final_df = final_df.dropna()  # or use fillna()

# -----------------------------
# STEP 6: SAVE FINAL DATASET
# -----------------------------
final_df.to_csv("final_merged_dataset.csv", index=False)

print("Merged dataset shape:", final_df.shape)
print(final_df.head())

Merged dataset shape: (19689, 16)
           crop  year       season location     area  production  fertilizer  \
0      Arecanut  1997  Whole Year     Assam  73814.0       56708  7024878.38   
1     Arhar/Tur  1997  Kharif         Assam   6637.0        4685   631643.29   
2   Castor seed  1997  Kharif         Assam    796.0          22    75755.32   
3      Coconut   1997  Whole Year     Assam  19656.0   126905000  1870661.52   
4  Cotton(lint)  1997  Kharif         Assam   1739.0         794   165500.63   

   pesticide   crop_yield  avg_temp_c  total_rainfall_mm  \
0   22882.34     0.796087       22.41            1468.92   
1    2057.47     0.710435       22.41            1468.92   
2     246.76     0.238333       22.41            1468.92   
3    6093.36  5238.051739       22.41            1468.92   
4     539.09     0.420909       22.41            1468.92   

   avg_humidity_percent   n   p   k  soil_ph  
0                 70.71  60  18  38      5.8  
1                 70.71  60  1

In [ ]:
!pip install xgboost

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error


from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("final_merged_dataset.csv")

# -----------------------------
# SELECT FEATURES & TARGET
# -----------------------------
# Adjust based on your dataset columns
X = df.drop(columns=['crop_yield'])   # features
y = df['crop_yield']                  # target

# -----------------------------
# HANDLE CATEGORICAL DATA
# -----------------------------
X = pd.get_dummies(X)

# -----------------------------
# TRAIN-TEST SPLIT (80/20)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# MODEL 1: RANDOM FOREST
# -----------------------------
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# -----------------------------
# MODEL 2: XGBOOST
# -----------------------------
xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

# -----------------------------
# EVALUATION FUNCTION
# -----------------------------


def evaluate(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))  # fixed

    print(f"\n{model_name} Performance:")
    print("MAE:", mae)
    print("RMSE:", rmse)
    
 

# -----------------------------
# EVALUATE BOTH MODELS
# -----------------------------
evaluate(y_test, rf_pred, "Random Forest")
evaluate(y_test, xgb_pred, "XGBoost")


Random Forest Performance:
MAE: 8.210297078463778
RMSE: 114.57660688485052

XGBoost Performance:
MAE: 13.294742568703903
RMSE: 237.126853065161


In [ ]:
!pip install --upgrade scikit-learn

In [2]:
print(xgb_pred)
print(rf_pred)

[2.2300353  1.2061541  7.8427505  ... 2.0429845  0.69231683 0.4983428 ]
[4.00229417 1.32727953 6.64987218 ... 1.41685796 0.69030972 0.35980009]


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create scaler
scaler = StandardScaler()

# Fit on training data only
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

In [4]:
rf.fit(X_train_scaled, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [5]:
xgb.fit(X_train_scaled, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [16]:
xgb_pred = xgb.predict(X_test_scaled)
rf_pred = rf.predict(X_test_scaled)
print(xgb_pred)
print(rf_pred)

[-0.41023275 15.213397    0.94270426 ... -1.4737964  -0.41023275
 -0.41023275]
[ 79.71797896 249.85998337  16.9927927  ...  79.68215654  83.69011653
  79.63513062]


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [17]:
def evaluate(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))  # fixed

    print(f"\n{model_name} Performance:")
    print("MAE:", mae)
    print("RMSE:", rmse)
    
 

# -----------------------------
# EVALUATE BOTH MODELS
# -----------------------------
evaluate(y_test, rf_pred, "Random Forest")
evaluate(y_test, xgb_pred, "XGBoost")


Random Forest Performance:
MAE: 153.68121411412187
RMSE: 862.8327145128615

XGBoost Performance:
MAE: 37.603615500097106
RMSE: 419.8457206323374


In [10]:
df

,crop,year,season,location,area,production,fertilizer,pesticide,crop_yield,avg_temp_c,total_rainfall_mm,avg_humidity_percent,n,p,k,soil_ph
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,7024878.38,22882.34,0.796087,22.41,1468.92,70.71,60,18,38,5.8
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,631643.29,2057.47,0.710435,22.41,1468.92,70.71,60,18,38,5.8
2,Castor seed,1997,Kharif,Assam,796.0,22,75755.32,246.76,0.238333,22.41,1468.92,70.71,60,18,38,5.8
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,1870661.52,6093.36,5238.051739,22.41,1468.92,70.71,60,18,38,5.8
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,165500.63,539.09,0.420909,22.41,1468.92,70.71,60,18,38,5.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19684,Small millets,1998,Kharif,Nagaland,4000.0,2000,395200.00,1160.00,0.500000,18.55,915.19,69.91,56,16,36,5.8
19685,Wheat,1998,Rabi,Nagaland,1000.0,3000,98800.00,290.00,3.000000,18.55,915.19,69.91,56,16,36,5.8
19686,Maize,1997,Kharif,Jammu and Kashmir,310883.0,440900,29586735.11,96373.73,1.285000,8.63,447.57,52.51,70,25,42,6.7
19687,Rice,1997,Kharif,Jammu and Kashmir,275746.0,5488,26242746.82,85481.26,0.016667,8.63,447.57,52.51,70,25,42,6.7


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create scaler
scaler = StandardScaler()

# Fit on training data only
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

In [12]:
rf.fit(X_train_scaled, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [13]:
xgb.fit(X_train_scaled, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [14]:
def evaluate(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))  # fixed

    print(f"\n{model_name} Performance:")
    print("MAE:", mae)
    print("RMSE:", rmse)
    
 

# -----------------------------
# EVALUATE BOTH MODELS
# -----------------------------
evaluate(y_test, rf_pred, "Random Forest")
evaluate(y_test, xgb_pred, "XGBoost")


Random Forest Performance:
MAE: 80.46492475691834
RMSE: 898.0032742230532

XGBoost Performance:
MAE: 429.9011858799813
RMSE: 941.1718762461579


In [20]:
# Random Forest + XGBoost
# =========================================

# -----------------------------
# IMPORT LIBRARIES
# -----------------------------
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# -----------------------------
# LOAD DATASET
# -----------------------------
df = pd.read_csv("final_merged_dataset.csv")

# -----------------------------
# VIEW DATA
# -----------------------------
print(df.head())
print(df.columns)

# -----------------------------
# CLEAN COLUMN NAMES
# -----------------------------
df.columns = df.columns.str.lower().str.strip()

# -----------------------------
# OPTIONAL:
# KEEP ONLY MAIZE & RICE
# -----------------------------
df['crop'] = df['crop'].str.lower().str.strip()

df = df[df['crop'].isin(['maize', 'rice'])]

# -----------------------------
# HANDLE MISSING VALUES
# -----------------------------
df = df.dropna()

# -----------------------------
# OPTIONAL:
# REMOVE EXTREME OUTLIERS
# -----------------------------
df = df[
    df['crop_yield'] <
    df['crop_yield'].quantile(0.99)
]

# -----------------------------
# FEATURES & TARGET
# -----------------------------
X = df.drop(columns=['crop_yield'])
y = df['crop_yield']

# -----------------------------
# CONVERT CATEGORICAL VARIABLES
# -----------------------------
X = pd.get_dummies(X)

# -----------------------------
# TRAIN-TEST SPLIT (80/20)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# NORMALIZATION
# -----------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================================
# RANDOM FOREST MODEL
# =========================================

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

# Train model
rf.fit(X_train_scaled, y_train)

# Predictions
rf_pred = rf.predict(X_test_scaled)

# =========================================
# XGBOOST MODEL
# =========================================

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Train model
xgb.fit(X_train_scaled, y_train)

# Predictions
xgb_pred = xgb.predict(X_test_scaled)

# =========================================
# EVALUATION FUNCTION
# =========================================

def evaluate(y_true, y_pred, model_name):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )


    print(f"\n{model_name} Performance")
    print("-" * 40)

    print("MAE :", mae)
    print("RMSE:", rmse)

# =========================================
# EVALUATE MODELS
# =========================================

evaluate(y_test, rf_pred, "Random Forest")

evaluate(y_test, xgb_pred, "XGBoost")

           crop  year       season location     area  production  fertilizer  \
0      Arecanut  1997  Whole Year     Assam  73814.0       56708  7024878.38   
1     Arhar/Tur  1997  Kharif         Assam   6637.0        4685   631643.29   
2   Castor seed  1997  Kharif         Assam    796.0          22    75755.32   
3      Coconut   1997  Whole Year     Assam  19656.0   126905000  1870661.52   
4  Cotton(lint)  1997  Kharif         Assam   1739.0         794   165500.63   

   pesticide   crop_yield  avg_temp_c  total_rainfall_mm  \
0   22882.34     0.796087       22.41            1468.92   
1    2057.47     0.710435       22.41            1468.92   
2     246.76     0.238333       22.41            1468.92   
3    6093.36  5238.051739       22.41            1468.92   
4     539.09     0.420909       22.41            1468.92   

   avg_humidity_percent   n   p   k  soil_ph  
0                 70.71  60  18  38      5.8  
1                 70.71  60  18  38      5.8  
2                

SyntaxError: invalid syntax (594784947.py, line 1)